In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 14:28:22.543156: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 14:28:23.723536: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [11]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

2023-07-02 14:28:28,104 [DEBUG] [Rain] Rain is initialized
2023-07-02 14:28:28,107 [DEBUG] [Provisioner] Creating coordinator
2023-07-02 14:28:28,109 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-02 14:28:28,111 [DEBUG] [LocalProvisioner] LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_sync()

2023-07-02 14:28:28,188 [DEBUG] [Rain] Creating workers
2023-07-02 14:28:28,211 [INFO] [Provisioner] provisioner is serving
2023-07-02 14:28:28,213 [DEBUG] [Provisioner] Starting coordinator
2023-07-02 14:28:28,219 [INFO] [Coordinator] coordinator is serving
2023-07-02 14:28:28,221 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-02 14:28:28,231 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-02 14:28:28,233 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-02 14:28:28,235 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-02 14:28:28,240 [INFO] [Worker] Worker is running on port: 50151
2023-07-02 14:28:28,244 [INFO] [Worker] Worker is running on port: 50152
2023-07-02 14:28:28,244 [INFO] [Worker] Worker is running on port: 50152
2023-07-02 14:28:28,249 [INFO] [Worker] Worker is running on port: 50153
2023-07-02 14:28:28,249 [INFO] [Worker] Worker is 

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 5s 18ms/step - loss: 0.6913 - accuracy: 0.7799
Epoch 2/2
155/157 [============================>.] - ETA: 0s - loss: 0.3062 - accuracy: 0.9076sending data to coordinator
sending data to coordinator
157/157 [==============================] - 3s 17ms/step - loss: 0.3057 - accuracy: 0.9077
sending data to coordinator


2023-07-02 14:30:30,651 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:30:30,654 [INFO] [Coordinator] thread 1 is done
2023-07-02 14:30:30,668 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:30:30,669 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:30:31,007 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_1_trained.pkl in coordinator
2023-07-02 14:30:31,009 [INFO] [Coordinator] thread 2 is done
2023-07-02 14:30:31,433 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_1_trained.pkl in coordinator
2023-07-02 14:30:31,438 [INFO] [Coordinator] thread 3 is done
2023-07-02 14:30:31,757 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_1_trained.pkl in coordinator
2023-07-02 14:30:32,032 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 14:30:32,312 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 4s 16ms/step - loss: 0.2559 - accuracy: 0.9239
Epoch 2/2
157/157 [==============================] - 4s 17ms/step - loss: 0.2544 - accuracy: 0.9216
Epoch 2/2
157/157 [==============================] - 4s 17ms/step - loss: 0.2511 - accuracy: 0.9259
Epoch 2/2
157/157 [==============================] - 2s 15ms/step - loss: 0.1967 - accuracy: 0.9406
sending data to coordinator
157/157 [==============================] - 2s 15ms/step - loss: 0.1988 - accuracy: 0.9376
sending data to coordinator
sending data to coordinator


2023-07-02 14:31:39,584 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:31:39,587 [INFO] [Coordinator] thread 1 is done
2023-07-02 14:31:39,610 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:31:39,621 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:31:39,884 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
2023-07-02 14:31:39,885 [INFO] [Coordinator] thread 2 is done
2023-07-02 14:31:40,167 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
2023-07-02 14:31:40,169 [INFO] [Coordinator] thread 3 is done
2023-07-02 14:31:40,525 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
2023-07-02 14:31:40,863 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 14:31:41,196 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 4s 17ms/step - loss: 0.1872 - accuracy: 0.9441
Epoch 2/2
157/157 [==============================] - 4s 17ms/step - loss: 0.1864 - accuracy: 0.9434
Epoch 2/2
157/157 [==============================] - 3s 17ms/step - loss: 0.1604 - accuracy: 0.9522


2023-07-02 14:32:50,023 [DEBUG] [Coordinator] coordinator received: Executed! from worker


sending data to coordinator
sending data to coordinator


2023-07-02 14:32:54,882 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:32:54,884 [INFO] [Coordinator] thread 1 is done
2023-07-02 14:32:55,176 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
2023-07-02 14:32:55,256 [DEBUG] [Coordinator] coordinator received: Executed! from worker
2023-07-02 14:32:55,260 [INFO] [Coordinator] thread 2 is done
2023-07-02 14:32:55,556 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
2023-07-02 14:32:55,559 [INFO] [Coordinator] thread 3 is done
2023-07-02 14:32:55,827 [DEBUG] [Coordinator] Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
2023-07-02 14:32:56,080 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 14:32:56,352 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023-07-02 14:32:56,626 [DEBUG] [Coordinator] coordinator received: Success! from divider
2023

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.0979 - accuracy: 0.9695

Test accuracy: 97.0%
